# Credit Card Fraud Detection Modelling.

## Load the train and test data sets.

In [1]:
import os
import pandas as pd

DATASETS_PATH = os.getenv("DATASETS_PATH")
DATASETS_PATH = DATASETS_PATH + "/AnomalyDetection/CreditCardFraudDetection/"

initial_train_df = pd.read_csv(DATASETS_PATH + "fraudTrain.csv")

initial_test_df = pd.read_csv(DATASETS_PATH + "fraudTest.csv")

initial_train_df["trans_date_trans_time"] = pd.to_datetime(initial_train_df["trans_date_trans_time"])
initial_test_df["trans_date_trans_time"] = pd.to_datetime(initial_test_df["trans_date_trans_time"])

### As can be seen from the EDA, the test data set is a temporal continuation of the train data set, so the time-series features should be calculated on a merged dataset, or necessery values should be taken from the train dataset for test dataset features.

### We are going to to build up features that describe "normal" behaviour of a customer. For that we are going to take temporal and spacial distributions and calculate Z-scores for them. By Z-Score a model is able to detect irregularities (deviation of a value from the normal distribution).

In [2]:
initial_train_df['minutes_of_day'] = initial_train_df['trans_date_trans_time'].dt.hour * 60 + initial_train_df['trans_date_trans_time'].dt.minute
initial_test_df['minutes_of_day'] = initial_test_df['trans_date_trans_time'].dt.hour * 60 + initial_test_df['trans_date_trans_time'].dt.minute

### Calculate a map of average number of minutes of the day for a purchase and its deviation for each credit card holder, using the train dataset.

In [3]:
df_temporal_stats = initial_train_df.groupby('cc_num')['minutes_of_day'].agg(['mean', 'std']).reset_index()
df_temporal_stats.columns = ['cc_num', 'mean_minutes_of_day', 'std_minutes_of_day']

initial_train_df = initial_train_df.merge(df_temporal_stats, on='cc_num', how='left')
initial_test_df = initial_test_df.merge(df_temporal_stats, on='cc_num', how='left')

initial_train_df['temporal_z_score'] = (initial_train_df['minutes_of_day'] - initial_train_df['mean_minutes_of_day']) / initial_train_df['std_minutes_of_day']
initial_test_df['temporal_z_score'] = (initial_test_df['minutes_of_day'] - initial_test_df['mean_minutes_of_day']) / initial_test_df['std_minutes_of_day']


In [4]:
print(initial_train_df)

         Unnamed: 0 trans_date_trans_time               cc_num  \
0                 0   2019-01-01 00:00:18     2703186189652095   
1                 1   2019-01-01 00:00:44         630423337322   
2                 2   2019-01-01 00:00:51       38859492057661   
3                 3   2019-01-01 00:01:16     3534093764340240   
4                 4   2019-01-01 00:03:06      375534208663984   
...             ...                   ...                  ...   
1296670     1296670   2020-06-21 12:12:08       30263540414123   
1296671     1296671   2020-06-21 12:12:19     6011149206456997   
1296672     1296672   2020-06-21 12:12:32     3514865930894695   
1296673     1296673   2020-06-21 12:13:36     2720012583106919   
1296674     1296674   2020-06-21 12:13:37  4292902571056973207   

                                    merchant       category     amt  \
0                 fraud_Rippin, Kub and Mann       misc_net    4.97   
1            fraud_Heller, Gutmann and Zieme    grocery_pos  107.

## Calculate the distribution of distances between merchants and a customer for all corresponding transactions.

In [5]:
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    # convert degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    r = 6371  # Earth radius in kilometers
    return c * r

initial_train_df["distance_km"] = haversine(initial_train_df["lat"], initial_train_df["long"], initial_train_df["merch_lat"], initial_train_df["merch_long"])
initial_test_df["distance_km"] = haversine(initial_test_df["lat"], initial_test_df["long"], initial_test_df["merch_lat"], initial_test_df["merch_long"])

df_spacial_stats = initial_train_df.groupby('cc_num')['distance_km'].agg(['mean', 'std']).reset_index()
df_spacial_stats.columns = ['cc_num', 'mean_distance_km', 'std_distance_km']

initial_train_df = initial_train_df.merge(df_spacial_stats, on='cc_num', how='left')
initial_test_df = initial_test_df.merge(df_spacial_stats, on='cc_num', how='left')

initial_train_df['spacial_z_score'] = (initial_train_df['distance_km'] - initial_train_df['mean_distance_km']) / initial_train_df['std_distance_km']
initial_test_df['spacial_z_score'] = (initial_test_df['distance_km'] - initial_test_df['mean_distance_km']) / initial_test_df['std_distance_km']

print(initial_train_df)


         Unnamed: 0 trans_date_trans_time               cc_num  \
0                 0   2019-01-01 00:00:18     2703186189652095   
1                 1   2019-01-01 00:00:44         630423337322   
2                 2   2019-01-01 00:00:51       38859492057661   
3                 3   2019-01-01 00:01:16     3534093764340240   
4                 4   2019-01-01 00:03:06      375534208663984   
...             ...                   ...                  ...   
1296670     1296670   2020-06-21 12:12:08       30263540414123   
1296671     1296671   2020-06-21 12:12:19     6011149206456997   
1296672     1296672   2020-06-21 12:12:32     3514865930894695   
1296673     1296673   2020-06-21 12:13:36     2720012583106919   
1296674     1296674   2020-06-21 12:13:37  4292902571056973207   

                                    merchant       category     amt  \
0                 fraud_Rippin, Kub and Mann       misc_net    4.97   
1            fraud_Heller, Gutmann and Zieme    grocery_pos  107.

## Recategorize the job descriptions for each customer.

In [6]:
unique_jobs = initial_train_df["job"].fillna("").astype(str).unique()

In [7]:
isco_df = pd.read_excel(DATASETS_PATH + "ISCO-08 EN Structure and definitions.xlsx")

In [8]:
isco_df["Title EN"] = isco_df["Title EN"].fillna("")
isco_df["Definition"] = isco_df["Definition"].fillna("")
isco_df["Tasks include"] = isco_df["Tasks include"].fillna("")

print(isco_df)

     Level  ISCO 08 Code                                           Title EN  \
0        1             1                                           Managers   
1        2            11  Chief Executives, Senior Officials and Legisla...   
2        3           111                   Legislators and Senior Officials   
3        4          1111                                        Legislators   
4        4          1112                        Senior Government Officials   
..     ...           ...                                                ...   
614      3            21             Non-commissioned Armed Forces Officers   
615      4           210             Non-commissioned Armed Forces Officers   
616      2             3              Armed Forces Occupations, Other Ranks   
617      3            31              Armed Forces Occupations, Other Ranks   
618      4           310              Armed Forces Occupations, Other Ranks   

                                            Definit

In [9]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def clean_text(series):
    return series.fillna("").astype(str)


initial_train_df["job"] = clean_text(initial_train_df["job"])
unique_jobs = initial_train_df["job"].unique().tolist()

isco_texts = (
    clean_text(isco_df["Title EN"]) + " " +
    clean_text(isco_df["Definition"]) + " " +
    clean_text(isco_df["Tasks include"])
).tolist()

model = SentenceTransformer("all-MiniLM-L6-v2")

job_embeddings = model.encode(unique_jobs, show_progress_bar=True)
isco_embeddings = model.encode(isco_texts, show_progress_bar=True)

similarity = cosine_similarity(job_embeddings, isco_embeddings)

best_match_idx = similarity.argmax(axis=1)
best_scores = similarity.max(axis=1)

mapping_df = pd.DataFrame({
    "job": unique_jobs,
    "isco_code": isco_df.iloc[best_match_idx]["ISCO 08 Code"].values.astype(int),
    "isco_title": isco_df.iloc[best_match_idx]["Title EN"].values,
    "similarity_score": best_scores
})

initial_train_df = initial_train_df.merge(mapping_df, on="job", how="left")

test_jobs = clean_text(initial_test_df["job"]).unique().tolist()

test_embeddings = model.encode(test_jobs, show_progress_bar=True)

similarity_test = cosine_similarity(test_embeddings, isco_embeddings)

best_idx_test = similarity_test.argmax(axis=1)

test_mapping = pd.DataFrame({
    "job": test_jobs,
    "isco_code": isco_df.iloc[best_idx_test]["ISCO 08 Code"].values,
    "isco_title": isco_df.iloc[best_idx_test]["Title EN"].values,
    "similarity_score": similarity_test.max(axis=1)
})

initial_test_df = initial_test_df.merge(test_mapping, on="job", how="left")

C:\WORKSPACE\Personal\Softwareentwicklung\DataSciencePortfolio\AnomalyDetection\CreditCardFraudDetection\CreditCardFraudDetectionVenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 7250.86it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|████████████████████████████████████████████████████████████████████████| 15/15 [00:00<00:00, 109.58it/s]


In [10]:
print(initial_train_df[["job","isco_code","isco_title","similarity_score"]])
print()
print(initial_test_df[["job","isco_code","isco_title","similarity_score"]])

                                       job  isco_code  \
0                Psychologist, counselling       2634   
1        Special educational needs teacher       2352   
2              Nature conservation officer       2133   
3                          Patent attorney        261   
4           Dance movement psychotherapist       2653   
...                                    ...        ...   
1296670                       Geoscientist       2114   
1296671   Production assistant, television       2654   
1296672                    Naval architect       3152   
1296673              Volunteer coordinator       5312   
1296674           Therapist, horticultural       9214   

                                              isco_title  similarity_score  
0                                          Psychologists          0.516353  
1                                 Special Needs Teachers          0.714934  
2                 Environmental Protection Professionals          0.421878  
3      

In [11]:
category_map = {v: i for i, v in enumerate(initial_train_df["category"].dropna().unique())}

initial_train_df["category_id"] = initial_train_df["category"].map(category_map)
initial_test_df["category_id"] = initial_test_df["category"].map(category_map).fillna(-1).astype(int)

In [12]:
print(initial_train_df[["category", "category_id"]])
print()
print(initial_test_df[["category", "category_id"]])
print()

              category  category_id
0             misc_net            0
1          grocery_pos            1
2        entertainment            2
3        gas_transport            3
4             misc_pos            4
...                ...          ...
1296670  entertainment            2
1296671    food_dining            8
1296672    food_dining            8
1296673    food_dining            8
1296674    food_dining            8

[1296675 rows x 2 columns]

              category  category_id
0        personal_care            9
1        personal_care            9
2       health_fitness           10
3             misc_pos            4
4               travel           11
...                ...          ...
555714  health_fitness           10
555715       kids_pets           12
555716       kids_pets           12
555717          travel           11
555718   entertainment            2

[555719 rows x 2 columns]



In [13]:
print(initial_test_df["isco_code"].isna().sum())

0


In [14]:
gender_map = {v: i for i, v in enumerate(initial_train_df["gender"].dropna().unique())}

initial_train_df["gender_id"] = initial_train_df["gender"].map(gender_map)
initial_test_df["gender_id"] = initial_test_df["gender"].map(gender_map).fillna(-1).astype(int)

In [15]:
print(initial_train_df[["gender","gender_id"]])
print()
print(initial_test_df[["gender","gender_id"]])

        gender  gender_id
0            F          0
1            F          0
2            M          1
3            M          1
4            M          1
...        ...        ...
1296670      M          1
1296671      M          1
1296672      M          1
1296673      M          1
1296674      M          1

[1296675 rows x 2 columns]

       gender  gender_id
0           M          1
1           F          0
2           F          0
3           M          1
4           M          1
...       ...        ...
555714      M          1
555715      M          1
555716      F          0
555717      M          1
555718      M          1

[555719 rows x 2 columns]


In [16]:
from datetime import date

def compute_age(df):
    df["dob"] = pd.to_datetime(df["dob"], errors="coerce")
    
    today = pd.Timestamp(date.today())
    
    age = (
        today.year - df["dob"].dt.year
        - (
            (today.month < df["dob"].dt.month) |
            ((today.month == df["dob"].dt.month) & (today.day < df["dob"].dt.day))
        )
    )
    
    return age.astype("Int64")

initial_train_df["age"] = compute_age(initial_train_df)
initial_test_df["age"] = compute_age(initial_test_df)

In [17]:
print(initial_train_df[["dob","age"]])
print()
print(initial_test_df[["dob","age"]])

               dob  age
0       1988-03-09   38
1       1978-06-21   47
2       1962-01-19   64
3       1967-01-12   59
4       1986-03-28   40
...            ...  ...
1296670 1961-11-24   64
1296671 1979-12-11   46
1296672 1967-08-30   58
1296673 1980-08-18   45
1296674 1995-08-16   30

[1296675 rows x 2 columns]

              dob  age
0      1968-03-19   58
1      1990-01-17   36
2      1970-10-21   55
3      1987-07-25   38
4      1955-07-06   70
...           ...  ...
555714 1966-02-13   60
555715 1999-12-27   26
555716 1981-11-29   44
555717 1965-12-15   60
555718 1993-05-10   32

[555719 rows x 2 columns]


## Calculate the purchase amount distribution for each person.

In [18]:
initial_train_df["log_amt"] = np.log1p(initial_train_df["amt"])
initial_test_df["log_amt"] = np.log1p(initial_test_df["amt"])

df_amount_stats = initial_train_df.groupby("cc_num")["log_amt"].agg(["mean", "std"]).reset_index()
df_amount_stats.columns = ["cc_num", "mean_log_amt", "std_log_amt"]

initial_train_df = initial_train_df.merge(df_amount_stats, on="cc_num", how="left")
initial_test_df = initial_test_df.merge(df_amount_stats, on="cc_num", how="left")

initial_train_df["amt_z_score"] = (
    (initial_train_df["log_amt"] - initial_train_df["mean_log_amt"]) /
    initial_train_df["std_log_amt"]
)

initial_test_df["amt_z_score"] = (
    (initial_test_df["log_amt"] - initial_test_df["mean_log_amt"]) /
    initial_test_df["std_log_amt"]
)

In [19]:
print(initial_train_df)

         Unnamed: 0 trans_date_trans_time               cc_num  \
0                 0   2019-01-01 00:00:18     2703186189652095   
1                 1   2019-01-01 00:00:44         630423337322   
2                 2   2019-01-01 00:00:51       38859492057661   
3                 3   2019-01-01 00:01:16     3534093764340240   
4                 4   2019-01-01 00:03:06      375534208663984   
...             ...                   ...                  ...   
1296670     1296670   2020-06-21 12:12:08       30263540414123   
1296671     1296671   2020-06-21 12:12:19     6011149206456997   
1296672     1296672   2020-06-21 12:12:32     3514865930894695   
1296673     1296673   2020-06-21 12:13:36     2720012583106919   
1296674     1296674   2020-06-21 12:13:37  4292902571056973207   

                                    merchant       category     amt  \
0                 fraud_Rippin, Kub and Mann       misc_net    4.97   
1            fraud_Heller, Gutmann and Zieme    grocery_pos  107.

In [20]:
train_features_df = initial_train_df[["cc_num","gender_id","age","isco_code","zip","city_pop","category_id","temporal_z_score","spacial_z_score","amt_z_score"]]

test_features_df = initial_test_df[["cc_num","gender_id","age","isco_code","zip","city_pop","category_id","temporal_z_score","spacial_z_score","amt_z_score"]]

train_labels_df = initial_train_df["is_fraud"]

test_labels_df = initial_test_df["is_fraud"]

print(train_features_df)

                      cc_num  gender_id  age  isco_code    zip  city_pop  \
0           2703186189652095          0   38       2634  28654      3495   
1               630423337322          0   47       2352  99160       149   
2             38859492057661          1   64       2133  83252      4154   
3           3534093764340240          1   59        261  59632      1939   
4            375534208663984          1   40       2653  24433        99   
...                      ...        ...  ...        ...    ...       ...   
1296670       30263540414123          1   64       2114  84735       258   
1296671     6011149206456997          1   46       2654  21790       100   
1296672     3514865930894695          1   58       3152  88325       899   
1296673     2720012583106919          1   45       5312  57756      1126   
1296674  4292902571056973207          1   30       9214  59871       218   

         category_id  temporal_z_score  spacial_z_score  amt_z_score  
0               

## Calculate anomaly score by Isolation Forest.

In [21]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    n_estimators=100,
    contamination=0.01,
    random_state=42
)

model.fit(train_features_df)

# anomaly score (higher = more normal in sklearn)
train_features_df["anomaly_score"] = model.decision_function(train_features_df)
test_features_df["anomaly_score"] = model.decision_function(test_features_df)

In [22]:
print(train_features_df)
print()
print(test_features_df)

                      cc_num  gender_id  age  isco_code    zip  city_pop  \
0           2703186189652095          0   38       2634  28654      3495   
1               630423337322          0   47       2352  99160       149   
2             38859492057661          1   64       2133  83252      4154   
3           3534093764340240          1   59        261  59632      1939   
4            375534208663984          1   40       2653  24433        99   
...                      ...        ...  ...        ...    ...       ...   
1296670       30263540414123          1   64       2114  84735       258   
1296671     6011149206456997          1   46       2654  21790       100   
1296672     3514865930894695          1   58       3152  88325       899   
1296673     2720012583106919          1   45       5312  57756      1126   
1296674  4292902571056973207          1   30       9214  59871       218   

         category_id  temporal_z_score  spacial_z_score  amt_z_score  \
0              